# BANKSY Human Colon Cancer final benchmark: K=21

This is a fixed-resolution, matched-K production notebook for the Human Colon Cancer Visium HD benchmark.

- Common input: all 131,592 observations and the supervisor-specified 2,000 HVGs.
- Common BANKSY representation: `lambda=0.8`, `scaled_gaussian`, `max_m=1`, 15/30 spatial neighbours and 20 PCs.
- Fixed Leiden configuration: resolution `1.1`, 50 neighbours, seed 1 and full convergence (`n_iterations=-1`).
- Strict output rule: figures and result files are saved only if the observed number of domains is exactly K=21.
- Fast execution: this notebook reuses the previously generated common BANKSY PCA representation. Resolution-search time, plotting and biological validation are kept outside the matched-K clustering runtime.

Outputs include domain labels/counts, a spatial-domain map, H&E overlay, marker-domain heatmap, marker spatial-expression panels, a paper summary figure, quantitative spatial summaries, runtime, peak memory and reproducibility metadata. It does not write another H5AD or Zarr store.


In [ ]:
%load_ext autotime
%matplotlib inline


In [ ]:
import json
import os
import platform
import sys
import threading
import time
from contextlib import contextmanager
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', '/private/tmp/mplconfig_banksy')

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import spatialdata as sd
from banksy.main import LeidenPartition
from matplotlib.colors import ListedColormap
from scipy import sparse
from scipy.spatial import cKDTree
from spatialdata.transformations import get_transformation
import leidenalg

process = psutil.Process()
step_times = []


@contextmanager
def record_step(name, sample_interval=0.10):
    start = time.perf_counter()
    rss_start = process.memory_info().rss / 1024**3
    peak = [rss_start]
    stop = threading.Event()

    def sample_memory():
        while not stop.wait(sample_interval):
            peak[0] = max(peak[0], process.memory_info().rss / 1024**3)

    sampler = threading.Thread(target=sample_memory, daemon=True)
    sampler.start()
    try:
        yield
    finally:
        stop.set()
        sampler.join()
        seconds = time.perf_counter() - start
        rss_end = process.memory_info().rss / 1024**3
        peak[0] = max(peak[0], rss_end)
        row = {
            'step': name,
            'seconds': seconds,
            'minutes': seconds / 60,
            'rss_start_gb': rss_start,
            'rss_end_gb': rss_end,
            'peak_rss_gb': peak[0],
            'rss_delta_gb': rss_end - rss_start,
        }
        step_times.append(row)
        print(
            f"{name}: {seconds / 60:.2f} min; "
            f"RSS={rss_end:.2f} GB; peak={peak[0]:.2f} GB"
        )


In [ ]:
DATA_DIR = Path('/Users/zhuzhengyang/Desktop/HDS_Dissertation/Human_Colon_Cancer_Visium_HD')
BANKSY_DIR = Path('/Users/zhuzhengyang/Desktop/HDS_Dissertation/BANKSY')
CACHE_PATH = (
    DATA_DIR / 'banksy_full_2000hvg_v1_outputs' /
    'human_colon_cancer_banksy_full_2000hvg_v1.h5ad'
)
COMMON_TIMING_PATH = (
    DATA_DIR / 'banksy_full_2000hvg_v1_outputs' /
    'human_colon_cancer_banksy_full_2000hvg_v1_step_timing.csv'
)
SDATA_PATH = DATA_DIR / 'Visium_HD_Human_Colon_Cancer_processed_clusters'
TABLE_KEY = 'manual_analysis_count_area_filtered'
IMAGE_KEY = 'Visium_HD_Human_Colon_Cancer_hires_image'
GLOBAL_CS = 'Visium_HD_Human_Colon_Cancer'
OUT_DIR = BANKSY_DIR / 'final_benchmark_outputs' / 'K21'
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_K = 21
RESOLUTION = 1.1
SEED = 1
NUM_NN = 50
N_ITERATIONS = -1
SPATIAL_METRIC_NEIGHBOURS = 15

MARKER_GENES = [
    'COL1A1', 'LYZ', 'C1QC', 'SPP1', 'SELENOP',
    'FCGBP', 'MUC2', 'REG1A', 'REG2B',
    'CEACAM6', 'CEACAM5', 'IGKC', 'PECAM1', 'TRAC', 'LCN2',
]

domain_colors = (
    list(plt.get_cmap('tab20').colors) +
    list(plt.get_cmap('tab20b').colors)
)[:TARGET_K]
domain_cmap = ListedColormap(domain_colors)

print('Target K:', TARGET_K)
print('Fixed resolution:', RESOLUTION)
print('Cache:', CACHE_PATH)
print('Outputs:', OUT_DIR)


## 1. Load the shared BANKSY representation


In [ ]:
with record_step('load_cached_banksy_representation'):
    cached = ad.read_h5ad(CACHE_PATH, backed='r')
    embedding = np.asarray(cached.obsm['X_banksy_pca'], dtype=np.float32)
    spatial_xy = np.asarray(cached.obsm['spatial'], dtype=np.float32)
    observation_ids = cached.obs_names.astype(str).copy()
    n_observations, n_pcs = embedding.shape

if n_pcs != 20:
    raise RuntimeError(f'Expected 20 BANKSY PCs, found {n_pcs}.')
if spatial_xy.shape != (n_observations, 2):
    raise RuntimeError('Cached spatial coordinates do not align with the BANKSY embedding.')

print('BANKSY PCA:', embedding.shape, embedding.dtype)
print('Spatial coordinates:', spatial_xy.shape)


## 2. Build the BANKSY-space SNN graph


In [ ]:
with record_step('build_banksy_snn_graph'):
    partitioner = LeidenPartition(
        embedding,
        num_nn=NUM_NN,
        nns_have_weights=True,
        compute_shared_nn=True,
        filter_shared_nn=True,
        shared_nn_max_rank=3,
        shared_nn_min_shared_nbrs=5,
        verbose=False,
    )


## 3. Run the single fixed-resolution K=21 partition


In [ ]:
with record_step('fixed_resolution_leiden_partition'):
    label, modularity = partitioner.partition(
        resolution=RESOLUTION,
        partition_metric=leidenalg.RBConfigurationVertexPartition,
        n_iterations=N_ITERATIONS,
        seed=SEED,
    )

raw_labels = np.asarray(label.dense, dtype=np.int32)
unique_labels = np.unique(raw_labels)
actual_k = int(unique_labels.size)

print('Resolution:', RESOLUTION)
print('Observed K:', actual_k)
print('Modularity:', float(modularity))

if actual_k != TARGET_K:
    raise RuntimeError(
        f'Expected exactly K={TARGET_K}, but resolution={RESOLUTION} produced K={actual_k}. '
        'No benchmark figures or labels have been saved.'
    )

# Compact labels to 0..K-1 without changing domain membership.
label_map = {old: new for new, old in enumerate(unique_labels)}
domain_codes = np.fromiter(
    (label_map[value] for value in raw_labels),
    dtype=np.int16,
    count=n_observations,
)
domain_names = np.asarray([str(value) for value in domain_codes], dtype=object)
domain_counts = pd.Series(domain_codes).value_counts().sort_index()
domain_counts.index.name = 'domain'
domain_counts.name = 'n_observations'

display(domain_counts.to_frame())


## 4. Save labels, domain map and spatial metrics


In [ ]:
prefix = f'human_colon_cancer_banksy_k{TARGET_K}'
labels_path = OUT_DIR / f'{prefix}_labels.csv'
counts_path = OUT_DIR / f'{prefix}_domain_counts.csv'
domain_map_path = OUT_DIR / f'{prefix}_spatial_domains.png'

pd.DataFrame(
    {'banksy_domain': domain_names},
    index=pd.Index(observation_ids, name='observation_id'),
).to_csv(labels_path)
domain_counts.to_csv(counts_path)

with record_step('spatial_metrics_and_domain_plot'):
    tree = cKDTree(spatial_xy)
    _, neighbour_index = tree.query(
        spatial_xy,
        k=SPATIAL_METRIC_NEIGHBOURS + 1,
        workers=-1,
    )
    neighbour_index = neighbour_index[:, 1:]
    neighbour_agreement = float(
        np.mean(domain_codes[:, None] == domain_codes[neighbour_index])
    )
    boundary_fraction = 1.0 - neighbour_agreement

    moran_values = []
    for domain in range(TARGET_K):
        x = (domain_codes == domain).astype(np.float64)
        z = x - x.mean()
        denominator = np.dot(z, z)
        if denominator == 0:
            continue
        numerator = np.sum(z[:, None] * z[neighbour_index])
        moran_values.append(
            (n_observations / neighbour_index.size) * numerator / denominator
        )
    mean_domain_morans_i = float(np.mean(moran_values))

    domain_size_cv = float(domain_counts.std(ddof=0) / domain_counts.mean())
    proportions = domain_counts.to_numpy(dtype=float) / n_observations
    normalized_domain_entropy = float(
        -(proportions * np.log(proportions)).sum() / np.log(TARGET_K)
    )

    fig, ax = plt.subplots(figsize=(8.2, 7.4))
    scatter = ax.scatter(
        spatial_xy[:, 0], spatial_xy[:, 1],
        c=domain_codes, cmap=domain_cmap,
        vmin=-0.5, vmax=TARGET_K - 0.5,
        s=0.45, linewidths=0, rasterized=True,
    )
    ax.invert_yaxis()
    ax.set_aspect('equal', adjustable='box')
    ax.set_title(f'BANKSY spatial domains (K={TARGET_K})')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    handles = [
        plt.Line2D([], [], marker='o', linestyle='', color=domain_colors[i],
                   label=str(i), markersize=5)
        for i in range(TARGET_K)
    ]
    ax.legend(
        handles=handles, title='Domain', loc='center left',
        bbox_to_anchor=(1.01, 0.5), frameon=False,
        ncol=1 if TARGET_K <= 14 else 2,
    )
    fig.savefig(domain_map_path, dpi=300, bbox_inches='tight')
    plt.show()

print('Neighbour agreement:', neighbour_agreement)
print('Mean one-vs-rest Moran I:', mean_domain_morans_i)
print('Domain-size CV:', domain_size_cv)
print('Saved:', labels_path)
print('Saved:', counts_path)
print('Saved:', domain_map_path)


## 5. Marker-gene validation and H&E alignment


In [ ]:
with record_step('load_marker_expression_and_he_image'):
    sdata = sd.read_zarr(SDATA_PATH)
    full_table = sdata.tables[TABLE_KEY]
    full_table.var_names_make_unique()

    available_markers = [gene for gene in MARKER_GENES if gene in full_table.var_names]
    missing_markers = [gene for gene in MARKER_GENES if gene not in full_table.var_names]
    if not available_markers:
        raise RuntimeError('None of the requested biological marker genes were found.')

    if np.array_equal(full_table.obs_names.astype(str), observation_ids):
        marker_adata = full_table[:, available_markers].copy()
    else:
        missing_observations = pd.Index(observation_ids).difference(full_table.obs_names.astype(str))
        if len(missing_observations):
            raise RuntimeError('Cached BANKSY observations do not align with the SpatialData table.')
        marker_adata = full_table[observation_ids.tolist(), available_markers].copy()

    marker_layer = 'normalized_no_log' if 'normalized_no_log' in marker_adata.layers else None
    marker_matrix = marker_adata.layers[marker_layer] if marker_layer else marker_adata.X
    if sparse.issparse(marker_matrix):
        marker_matrix = marker_matrix.toarray()
    marker_matrix = np.asarray(marker_matrix, dtype=np.float32)

    image_da = sdata.images[IMAGE_KEY].transpose('y', 'x', 'c')
    he_image = np.asarray(image_da.values)
    transform = get_transformation(
        sdata.images[IMAGE_KEY],
        to_coordinate_system=GLOBAL_CS,
    )
    scale_y = float(transform.scale[transform.axes.index('y')])
    scale_x = float(transform.scale[transform.axes.index('x')])
    spatial_hires = np.column_stack([
        spatial_xy[:, 0] / scale_x,
        spatial_xy[:, 1] / scale_y,
    ])

print('Marker layer:', marker_layer)
print('Available markers:', available_markers)
print('Missing markers:', missing_markers)
print('H&E:', he_image.shape)


In [ ]:
with record_step('marker_validation_plots'):
    marker_frame = pd.DataFrame(
        marker_matrix,
        index=observation_ids,
        columns=available_markers,
    )
    marker_frame['banksy_domain'] = domain_names
    marker_means = marker_frame.groupby(
        'banksy_domain', sort=False, observed=True
    )[available_markers].mean()
    marker_means = marker_means.reindex([str(i) for i in range(TARGET_K)])
    marker_scale = marker_means.std(axis=0, ddof=0).replace(0, np.nan)
    marker_zscores = (
        (marker_means - marker_means.mean(axis=0)) / marker_scale
    ).fillna(0)

    marker_means_path = OUT_DIR / f'{prefix}_marker_means.csv'
    marker_zscores_path = OUT_DIR / f'{prefix}_marker_domain_zscores.csv'
    heatmap_path = OUT_DIR / f'{prefix}_marker_domain_heatmap.png'
    marker_panels_path = OUT_DIR / f'{prefix}_marker_spatial_panels.png'
    marker_means.to_csv(marker_means_path)
    marker_zscores.to_csv(marker_zscores_path)

    fig, ax = plt.subplots(
        figsize=(max(10, 0.72 * len(available_markers)), max(4, 0.40 * TARGET_K))
    )
    image = ax.imshow(
        marker_zscores.to_numpy(), aspect='auto',
        cmap='RdBu_r', vmin=-2, vmax=2,
    )
    ax.set_xticks(np.arange(len(available_markers)))
    ax.set_xticklabels(available_markers, rotation=45, ha='right')
    ax.set_yticks(np.arange(TARGET_K))
    ax.set_yticklabels(marker_zscores.index)
    ax.set_xlabel('Marker gene')
    ax.set_ylabel('BANKSY domain')
    ax.set_title(f'Marker enrichment by BANKSY domain (K={TARGET_K})')
    fig.colorbar(image, ax=ax, label='Standardized domain mean')
    fig.savefig(heatmap_path, dpi=300, bbox_inches='tight')
    plt.show()

    ncols = 4
    nrows = int(np.ceil(len(available_markers) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3.7 * nrows))
    axes = np.asarray(axes).ravel()
    for index, gene in enumerate(available_markers):
        values = np.log1p(np.maximum(marker_matrix[:, index], 0))
        vmax = float(np.quantile(values, 0.99))
        axes[index].scatter(
            spatial_xy[:, 0], spatial_xy[:, 1], c=values,
            cmap='Reds', vmin=0, vmax=max(vmax, 1e-6),
            s=0.18, linewidths=0, rasterized=True,
        )
        axes[index].invert_yaxis()
        axes[index].set_aspect('equal', adjustable='box')
        axes[index].set_title(gene)
        axes[index].set_xticks([])
        axes[index].set_yticks([])
    for ax in axes[len(available_markers):]:
        ax.axis('off')
    fig.suptitle(f'Colon marker expression used to validate BANKSY K={TARGET_K}', y=1.01)
    fig.tight_layout()
    fig.savefig(marker_panels_path, dpi=300, bbox_inches='tight')
    plt.show()

print('Saved:', marker_means_path)
print('Saved:', marker_zscores_path)
print('Saved:', heatmap_path)
print('Saved:', marker_panels_path)


In [ ]:
with record_step('he_overlay_and_paper_summary'):
    padding = 60
    x0 = max(0, int(np.floor(spatial_hires[:, 0].min())) - padding)
    x1 = min(he_image.shape[1], int(np.ceil(spatial_hires[:, 0].max())) + padding)
    y0 = max(0, int(np.floor(spatial_hires[:, 1].min())) - padding)
    y1 = min(he_image.shape[0], int(np.ceil(spatial_hires[:, 1].max())) + padding)
    he_crop = he_image[y0:y1, x0:x1]
    plot_xy = spatial_hires - np.array([x0, y0], dtype=np.float32)

    overlay_path = OUT_DIR / f'{prefix}_domains_on_he.png'
    paper_figure_path = OUT_DIR / f'{prefix}_paper_summary.png'

    fig, axes = plt.subplots(1, 2, figsize=(14, 6.3), constrained_layout=True)
    axes[0].imshow(he_crop, origin='upper')
    axes[0].set_title('H&E morphology')
    axes[1].imshow(he_crop, origin='upper', alpha=0.38)
    axes[1].scatter(
        plot_xy[:, 0], plot_xy[:, 1],
        c=domain_codes, cmap=domain_cmap,
        vmin=-0.5, vmax=TARGET_K - 0.5,
        s=0.45, linewidths=0, alpha=0.80, rasterized=True,
    )
    axes[1].set_title(f'BANKSY domains over H&E (K={TARGET_K})')
    for ax in axes:
        ax.set_xlim(0, he_crop.shape[1])
        ax.set_ylim(he_crop.shape[0], 0)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_aspect('equal')
        for spine in ax.spines.values():
            spine.set_visible(False)
    fig.savefig(overlay_path, dpi=300, bbox_inches='tight')
    plt.show()

    fig = plt.figure(figsize=(21, 7), constrained_layout=True)
    grid = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1.25])
    ax0 = fig.add_subplot(grid[0, 0])
    ax1 = fig.add_subplot(grid[0, 1])
    ax2 = fig.add_subplot(grid[0, 2])

    ax0.scatter(
        spatial_xy[:, 0], spatial_xy[:, 1], c=domain_codes,
        cmap=domain_cmap, vmin=-0.5, vmax=TARGET_K - 0.5,
        s=0.35, linewidths=0, rasterized=True,
    )
    ax0.invert_yaxis()
    ax0.set_aspect('equal')
    ax0.set_title(f'BANKSY domains (K={TARGET_K})')
    ax0.set_xticks([])
    ax0.set_yticks([])

    ax1.imshow(he_crop, origin='upper', alpha=0.42)
    ax1.scatter(
        plot_xy[:, 0], plot_xy[:, 1], c=domain_codes,
        cmap=domain_cmap, vmin=-0.5, vmax=TARGET_K - 0.5,
        s=0.32, linewidths=0, alpha=0.78, rasterized=True,
    )
    ax1.set_xlim(0, he_crop.shape[1])
    ax1.set_ylim(he_crop.shape[0], 0)
    ax1.set_aspect('equal')
    ax1.set_title('Domains aligned to H&E')
    ax1.set_xticks([])
    ax1.set_yticks([])

    heat = ax2.imshow(
        marker_zscores.to_numpy(), aspect='auto',
        cmap='RdBu_r', vmin=-2, vmax=2,
    )
    ax2.set_xticks(np.arange(len(available_markers)))
    ax2.set_xticklabels(available_markers, rotation=45, ha='right')
    ax2.set_yticks(np.arange(TARGET_K))
    ax2.set_yticklabels(marker_zscores.index)
    ax2.set_xlabel('Marker gene')
    ax2.set_ylabel('Domain')
    ax2.set_title('Marker enrichment')
    fig.colorbar(heat, ax=ax2, label='Standardized domain mean', shrink=0.75)
    fig.savefig(paper_figure_path, dpi=300, bbox_inches='tight')
    plt.show()

print('Saved:', overlay_path)
print('Saved:', paper_figure_path)


## 6. Save benchmark summary and reproducibility record


In [ ]:
timing_df = pd.DataFrame(step_times)
clustering_steps = ['build_banksy_snn_graph', 'fixed_resolution_leiden_partition']
matched_k_runtime_seconds = float(
    timing_df.loc[timing_df['step'].isin(clustering_steps), 'seconds'].sum()
)

domain_count_values = domain_counts.to_numpy(dtype=int)
summary = {
    'method': 'BANKSY',
    'target_k': TARGET_K,
    'actual_k': actual_k,
    'resolution': RESOLUTION,
    'seed': SEED,
    'leiden_iterations': N_ITERATIONS,
    'n_observations': n_observations,
    'n_hvgs_in_cached_representation': 2000,
    'banksy_lambda': 0.8,
    'banksy_k_geom_m0': 15,
    'banksy_k_geom_m1': 30,
    'banksy_max_m': 1,
    'banksy_pcs': n_pcs,
    'banksy_space_num_nn': NUM_NN,
    'modularity': float(modularity),
    'spatial_neighbour_agreement': neighbour_agreement,
    'spatial_boundary_fraction': boundary_fraction,
    'mean_domain_morans_i': mean_domain_morans_i,
    'domain_size_cv': domain_size_cv,
    'normalized_domain_entropy': normalized_domain_entropy,
    'smallest_domain_n': int(domain_count_values.min()),
    'largest_domain_n': int(domain_count_values.max()),
    'matched_k_clustering_seconds': matched_k_runtime_seconds,
    'matched_k_clustering_minutes': matched_k_runtime_seconds / 60,
    'workflow_seconds_including_validation_plots': float(timing_df['seconds'].sum()),
    'peak_rss_gb': float(timing_df['peak_rss_gb'].max()),
    'runtime_scope': 'SNN graph plus fixed-K Leiden from cached common BANKSY PCA',
    'common_banksy_representation_reused': True,
    'common_representation_timing_reference': str(COMMON_TIMING_PATH),
}

timing_path = OUT_DIR / f'{prefix}_timing.csv'
summary_path = OUT_DIR / f'{prefix}_benchmark_summary.csv'
metadata_path = OUT_DIR / f'{prefix}_metadata.json'

timing_df.to_csv(timing_path, index=False)
pd.DataFrame([summary]).to_csv(summary_path, index=False)

try:
    pybanksy_version = version('pybanksy')
except PackageNotFoundError:
    pybanksy_version = 'local source checkout'

metadata = {
    'method': 'BANKSY',
    'target_k': TARGET_K,
    'resolution': RESOLUTION,
    'input_cache': str(CACHE_PATH),
    'input_spatialdata': str(SDATA_PATH),
    'table_key': TABLE_KEY,
    'image_key': IMAGE_KEY,
    'available_markers': available_markers,
    'missing_markers': missing_markers,
    'marker_layer': marker_layer,
    'parameters': {
        'lambda': 0.8,
        'k_geom_m0': 15,
        'k_geom_m1': 30,
        'max_m': 1,
        'pca_dims': 20,
        'num_nn': NUM_NN,
        'seed': SEED,
        'leiden_iterations': N_ITERATIONS,
    },
    'runtime_scope_note': (
        'This matched-K notebook reuses the common BANKSY PCA representation. '
        'Do not report its clustering-only time as the complete end-to-end BANKSY runtime.'
    ),
    'python_version': sys.version,
    'platform': platform.platform(),
    'pybanksy_version': pybanksy_version,
    'anndata_version': ad.__version__,
    'spatialdata_version': sd.__version__,
    'large_h5ad_or_zarr_written': False,
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')

display(pd.DataFrame([summary]).T.rename(columns={0: 'value'}))
display(timing_df)
print('Saved:', timing_path)
print('Saved:', summary_path)
print('Saved:', metadata_path)
print('All outputs:', OUT_DIR)
